In [1]:
from pyspark.sql import SparkSession
from minio import Minio

spark = SparkSession.builder \
    .appName("TipPrediction") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Spark uses Hadoop configs internally to connect to storage systems like S3 / MinIO
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin123")
hadoop_conf.set("fs.s3a.path.style.access", "true")        # Required for MinIO because it uses path-style URLs instead of virtual-hosted style
                                                           # Example path-style: http://minio:9000/bucket-name/object
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")  # MinIO is running on HTTP locally (not HTTPS)

In [2]:
# ---------- Connect to MinIO ----------

minio_client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False
)

bucket_name = "nyc-taxi"

# Print all objects present inside the bucket
# This helps verify whether parquet files exist and are accessible
print("\n" + "="*60)
print(f"Objects inside '{bucket_name}':")
for obj in minio_client.list_objects(bucket_name, recursive=True):
    print(f" - {obj.object_name} ({obj.size} bytes)")
print("="*60)
print("NYC Taxi Parquet Upload Complete")
print("="*60)


Objects inside 'nyc-taxi':
 - 2023/yellow_tripdata_2023-01.parquet (47673370 bytes)
 - 2023/yellow_tripdata_2023-02.parquet (47748012 bytes)
 - 2023/yellow_tripdata_2023-03.parquet (56127762 bytes)
 - 2023/yellow_tripdata_2023-04.parquet (54222699 bytes)
 - 2023/yellow_tripdata_2023-05.parquet (58654627 bytes)
 - 2023/yellow_tripdata_2023-06.parquet (54999465 bytes)
 - 2023/yellow_tripdata_2023-07.parquet (48361828 bytes)
 - 2023/yellow_tripdata_2023-08.parquet (48152353 bytes)
 - 2023/yellow_tripdata_2023-09.parquet (47895515 bytes)
 - 2023/yellow_tripdata_2023-10.parquet (59009059 bytes)
 - 2023/yellow_tripdata_2023-11.parquet (56094653 bytes)
 - 2023/yellow_tripdata_2023-12.parquet (56804275 bytes)
 - 2024/yellow_tripdata_2024-01.parquet (49961641 bytes)
 - 2024/yellow_tripdata_2024-02.parquet (50349284 bytes)
 - 2024/yellow_tripdata_2024-03.parquet (60078280 bytes)
 - 2024/yellow_tripdata_2024-04.parquet (59133625 bytes)
 - 2024/yellow_tripdata_2024-05.parquet (62553128 bytes)
 - 

In [3]:
from pyspark.sql.functions import col

# Month 1 (January) has a different schema than the other files. Therefore, some of its columns' data types have been changed.
# Also, the Airport_fee column name was different, so it has been changed as well.

# Read January separately and cast to match the standard schema
df_jan = spark.read.parquet("s3a://nyc-taxi/2023/yellow_tripdata_2023-01.parquet")

df_jan = df_jan \
    .withColumn("VendorID", col("VendorID").cast("integer")) \
    .withColumn("passenger_count", col("passenger_count").cast("long")) \
    .withColumn("RatecodeID", col("RatecodeID").cast("long")) \
    .withColumn("PULocationID", col("PULocationID").cast("integer")) \
    .withColumn("DOLocationID", col("DOLocationID").cast("integer")) \
    .withColumnRenamed("airport_fee", "Airport_fee")

# Read the rest (Feb-Dec)
df_rest = spark.read.parquet(
    "s3a://nyc-taxi/2023/yellow_tripdata_2023-0[2-9].parquet",
    "s3a://nyc-taxi/2023/yellow_tripdata_2023-1[0-2].parquet"
)

# Union them together
df1 = df_jan.union(df_rest)

In [4]:
# Combining parquet files from all the folders to create a single dataframe.

from pyspark.sql.functions import lit

df2 = spark.read.parquet("s3a://nyc-taxi/2024/*.parquet")
df_1_2 = df1.union(df2)

# Newer congestion charge (column name - cbd_congestion_fee) was introduced in 2025's datasets. 
# So, to union it with previous years' dataframes, we created the same column in previous years' dataframes.

df_1_2 = df_1_2.withColumn('cbd_congestion_fee', lit(0.0))
df3 = spark.read.parquet("s3a://nyc-taxi/2025/*.parquet")
df = df_1_2.union(df3)
df.count()

111036384

# EDA

In [31]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+-----------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|     total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------

We can see three problems with the data through the summary statistics - 
1. Many rows don't contain any value in the passenger count column.
2. The min values of fees and amount-related columns are negative.
3. The max values for the trip_distance and fare_amount columns are too high.
4. Zero trip_distance and fare_amount values.

### Going through the rows with null values in the passenger count column.

In [32]:
from pyspark.sql.functions import col

df_null_passenger = df.filter(col("passenger_count").isNull())
df_null_passenger.describe().show()

+-------+------------------+---------------+------------------+----------+------------------+------------------+------------------+------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+--------------------+-----------+------------------+
|summary|          VendorID|passenger_count|     trip_distance|RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|payment_type|       fare_amount|             extra|           mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+-------+------------------+---------------+------------------+----------+------------------+------------------+------------------+------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+--------------------+-----------+------------------+
|

In [33]:
df_null_passenger.groupBy("VendorID").count().show()

+--------+--------+
|VendorID|   count|
+--------+--------+
|       1| 2391564|
|       2|10333323|
|       6|   19291|
+--------+--------+



In [34]:
df_null_passenger.groupBy("PULocationID").count().show()

+------------+------+
|PULocationID| count|
+------------+------+
|         148|246081|
|         243| 30060|
|          31|   923|
|         137|180374|
|          85| 13129|
|          65| 43493|
|         255| 84605|
|          53|  2760|
|         133|  8166|
|          78|  6104|
|         155|  6324|
|         108|  2026|
|         211|137590|
|         193| 15606|
|          34|  6386|
|         101|  2033|
|         126|  5921|
|          81|  3948|
|          28|  9754|
|         210|  4470|
+------------+------+
only showing top 20 rows



In [35]:
df_null_passenger.groupBy("DOLocationID").count().show()

+------------+------+
|DOLocationID| count|
+------------+------+
|         148|236461|
|         243| 85820|
|          31|  1774|
|         137|179076|
|          85| 13306|
|          65| 22668|
|         255| 41330|
|          53|  6211|
|         133|  7503|
|          78|  7578|
|         155|  7884|
|         108|  3590|
|         211|125434|
|         193| 11754|
|          34|  4322|
|         126|  6356|
|         101|  2569|
|          81|  4304|
|          28|  9795|
|         210|  6891|
+------------+------+
only showing top 20 rows



This shows there is not a pattern with null values for a specific vendor or location. Therefore we can go ahead with removing these rows with null values.

In [5]:
df = df.dropna()
df.count()

98292206

### For tipping prediction/analysis, negative amounts are not normal “real trips”, so we are going to exclude them

In [6]:
df = df.filter(
    (col("fare_amount") >= 0) &
    (col("trip_distance") >= 0) &
    (col("total_amount") >= 0) &
    (col("tip_amount") >= 0) &
    (col("extra") >= 0) &
    (col("mta_tax") >= 0) &
    (col("tolls_amount") >= 0) &
    (col("improvement_surcharge") >= 0) &
    (col("congestion_surcharge") >= 0) &
    (col("Airport_fee") >= 0) &
    (col("cbd_congestion_fee") >= 0)
)

# df.describe().show()

### Now, we will do the outlier analysis for trip distance

In [60]:
df.select("trip_distance").describe().show()

+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|          96746457|
|   mean|3.5408033883862964|
| stddev| 76.14819849332225|
|    min|               0.0|
|    max|          161726.1|
+-------+------------------+



In [61]:
q1, median, q3 = df.approxQuantile("trip_distance", [0.25, 0.5, 0.75], 0.01)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Median:", median)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

Q1: 1.02
Median: 1.72
Q3: 3.28
IQR: 2.26
Lower Bound: -2.3699999999999997
Upper Bound: 6.67


In [62]:
from pyspark.sql.functions import col

outliers_df = df.filter((col("trip_distance") < lower) | (col("trip_distance") > upper))
normal_df   = df.filter((col("trip_distance") >= lower) & (col("trip_distance") <= upper))

print("Normal Rows:", normal_df.count())
print("Outlier Rows:", outliers_df.count())

Normal Rows: 83708873
Outlier Rows: 13037584


The IQR method won't be applicable here since the outlier rows are in millions. So, we will take 100 as the outlier distance.

In [66]:
long_distance_df = df.filter(col("trip_distance") > 100)

print("Long Distance Rows:", long_distance_df.count())

High Distance Rows: 2012


In [67]:
total_rows = df.count()
long_distance_rows = long_distance_df.count()

print("Long Distance Rows %:", (long_distance_rows / total_rows) * 100)

High Distance Rows %: 0.0020796627208787607


100 seems like a good value to take as an outlier. Therefore, we will filter out the rows that contain a trip distance greater than 100.

In [7]:
df = df.filter((col("trip_distance") <= 100))

print("Rows after trimming:", df.count())

Rows after trimming: 96744445


In [70]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|           mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+-------

We can still see very high total amount values. Next, we will filter for that (i.e., amount greater than 500).  

In [73]:
high_cost_df = df.filter(col("total_amount") > 500)
print("High Cost Rows:", high_cost_df.count())

High Cost Rows: 1761


In [74]:
total_rows = df.count()
high_cost_df = high_cost_df.count()

print("High Cost Rows %:", (high_cost_df / total_rows) * 100)

High Cost Rows %: 0.0018202595508196879


In [8]:
df = df.filter((col("total_amount") <= 500))

print("Rows after trimming:", df.count())

Rows after trimming: 96742684


In [76]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|      fare_amount|             extra|             mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+------------------+------------------+---------------------+----

### Check for zero distance and zero fare rows.

In [81]:
zero_distance_df = df.filter(col("trip_distance") == 0)
print("Zero Distance Rows:", zero_distance_df.count())

Zero Distance Rows: 1179794


In [82]:
zero_fare_df = df.filter(col("fare_amount") == 0)
print("Zero Fare Rows:", zero_fare_df.count())

Zero Fare Rows: 30019


In [86]:
zero_distance_df.select("fare_amount", "total_amount", "tip_amount", "payment_type") \
  .describe() \
  .show()

+-------+------------------+-----------------+-----------------+------------------+
|summary|       fare_amount|     total_amount|       tip_amount|      payment_type|
+-------+------------------+-----------------+-----------------+------------------+
|  count|           1179794|          1179794|          1179794|           1179794|
|   mean|30.403765801486582|37.93141550134817|3.454934920842125|  1.67966950162486|
| stddev| 39.06620980276862|42.90768346177385|7.665664297335325|0.9476936000172407|
|    min|               0.0|              0.0|              0.0|                 1|
|    max|             500.0|            500.0|            494.0|                 5|
+-------+------------------+-----------------+-----------------+------------------+



In [88]:
zero_fare_df.select("trip_distance", "fare_amount", "total_amount", "tip_amount", "payment_type") \
  .describe() \
  .show()

+-------+------------------+-----------+-----------------+------------------+------------------+
|summary|     trip_distance|fare_amount|     total_amount|        tip_amount|      payment_type|
+-------+------------------+-----------+-----------------+------------------+------------------+
|  count|             30019|      30019|            30019|             30019|             30019|
|   mean|1.9949012292214916|        0.0|2.604796295679404|0.7418185149405377|2.5519171191578667|
| stddev|  5.50628341973777|        0.0|9.404846416926187| 8.269101898021075|1.0091499106679318|
|    min|               0.0|        0.0|              0.0|               0.0|                 1|
|    max|              93.5|        0.0|           286.43|             270.0|                 5|
+-------+------------------+-----------+-----------------+------------------+------------------+



Because trip_distance and fare_amount are important features, and 0-distance or 0-fare-amount rows can confuse our analysis. We will filter them out.

In [9]:
df = df.filter((col("trip_distance") > 0) & (col("fare_amount") > 0))
df.count()

95547339

In [13]:
df.describe().show()

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|     trip_distance|       RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+----

### The dataset is clean and ready to be used for modelling.

Next, we will create the df_tips1 dataframe to include metrics such as pickup_hour, is_weekend, etc. that will help us in determining the average tip amount.

In [10]:
from pyspark.sql.functions import (
    hour, dayofweek, month, when, sqrt, 
    unix_timestamp, cos, sin, radians, concat
)

df_tips1 = df.select("tip_amount", "fare_amount", "trip_distance", "payment_type", "PULocationID", "DOLocationID", "tpep_pickup_datetime", "tpep_dropoff_datetime")

df_tips1 = df_tips1.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
       .withColumn("pickup_day", dayofweek("tpep_pickup_datetime")) \
       .withColumn("pickup_month", month("tpep_pickup_datetime")) \
       .withColumn("is_weekend", when(col("pickup_day").isin([1, 7]), 1).otherwise(0)) \
       .withColumn("is_rush_hour", when(
           ((col("pickup_hour") >= 6) & (col("pickup_hour") <= 10)) | 
           ((col("pickup_hour") >= 16) & (col("pickup_hour") <= 20)), 1
       ).otherwise(0)) \
       .withColumn("is_night", when(
           ((col("pickup_hour") >= 20) | (col("pickup_hour") <= 6)), 1
       ).otherwise(0)) \
       .withColumn("tip_percent", (col("tip_amount") / col("fare_amount")) * 100) \
       .withColumn("PU_DO_pair", concat(col("PULocationID").cast("string"), lit("_"), col("DOLocationID").cast("string")))

In [16]:
df_tips1.show()

+----------+-----------+-------------+------------+------------+------------+--------------------+---------------------+-----------+----------+------------+----------+------------+--------+------------------+----------+
|tip_amount|fare_amount|trip_distance|payment_type|PULocationID|DOLocationID|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_hour|pickup_day|pickup_month|is_weekend|is_rush_hour|is_night|       tip_percent|PU_DO_pair|
+----------+-----------+-------------+------------+------------+------------+--------------------+---------------------+-----------+----------+------------+----------+------------+--------+------------------+----------+
|       0.0|        9.3|         0.97|           2|         161|         141| 2023-01-01 00:32:10|  2023-01-01 00:40:36|          0|         1|           1|         1|           0|       0|               0.0|   161_141|
|       4.0|        7.9|          1.1|           1|          43|         237| 2023-01-01 00:55:08|  2023-01-01 01:01:27|

The payment_type 2 (Cash) consistently shows a tip_amount of 0.0. This isn't necessarily because passengers didn't tip, but because cash tips aren't recorded in the system. Likewise, other payment systems are also not much helpful here. Therefore, we need to filter them out.

In [11]:
df_tips1 = df_tips1.filter(col("payment_type") == 1)

In [12]:
df_tips1.count()

79757391

Before moving ahead with the columns to aggregate data based on, let's take a look at how much they effect the avg tip percent data.

In [19]:
df_tips1.groupBy('is_night').avg('tip_percent').show()

+--------+------------------+
|is_night|  avg(tip_percent)|
+--------+------------------+
|       1|25.652914971662863|
|       0| 25.45770167729256|
+--------+------------------+



In [20]:
df_tips1.groupBy('is_rush_hour').avg('tip_percent').show()

+------------+------------------+
|is_rush_hour|  avg(tip_percent)|
+------------+------------------+
|           1|25.954784225899857|
|           0|25.088459399006698|
+------------+------------------+



In [21]:
df_tips1.groupBy('is_weekend').avg('tip_percent').show()

+----------+------------------+
|is_weekend|  avg(tip_percent)|
+----------+------------------+
|         1|24.828555103618953|
|         0| 25.76748909972411|
+----------+------------------+



In [22]:
df_tips1.groupBy('pickup_month').avg('tip_percent').show()

+------------+------------------+
|pickup_month|  avg(tip_percent)|
+------------+------------------+
|          12|25.094430361089938|
|           1| 26.17986607994806|
|          10|  25.1045627233786|
|           2|25.640637133914066|
|           9|24.462165139128004|
|          11|25.838064580083504|
|           6| 25.39009100305301|
|           5|25.176676371847293|
|           4|25.833125426733066|
|           8|25.085676363384053|
|           7| 26.04541650619972|
|           3|25.861159713259585|
+------------+------------------+



In [23]:
df_tips1.groupBy('pickup_day').avg('tip_percent').show()

+----------+------------------+
|pickup_day|  avg(tip_percent)|
+----------+------------------+
|         1| 24.68717291108004|
|         6|25.410508234716264|
|         3| 25.90459606725041|
|         5|26.139571511754323|
|         4| 25.62878622528968|
|         7| 24.94809902633729|
|         2|25.726623277243664|
+----------+------------------+



In [24]:
df_tips1.groupBy('pickup_hour').avg('tip_percent').show()

+-----------+------------------+
|pickup_hour|  avg(tip_percent)|
+-----------+------------------+
|         12|24.321234681388066|
|         22|25.501986294332305|
|          1|25.076702127132897|
|         13|24.750965118406594|
|          6|26.507472521438352|
|         16|26.181031922599008|
|          3|25.194811575017702|
|         20|25.841446947656554|
|          5| 23.15155688651097|
|         19| 27.32282026567797|
|         15|24.898389362389395|
|          9|  24.4288315248703|
|         17|26.509070491832205|
|          4|28.874924322051587|
|          8| 24.52653518680321|
|         23|25.519452570890802|
|          7|24.571234304755865|
|         10|  24.6742172351314|
|         21| 25.66991246508552|
|         11|25.246506511408157|
+-----------+------------------+
only showing top 20 rows



In [12]:
from pyspark.sql.functions import avg, max as spark_max, min as spark_min

def effect_size(df, group_col, target="tip_percent"):
    ag = df.groupBy(group_col).agg(avg(target).alias("avg_tip"))
    return ag.agg((spark_max("avg_tip") - spark_min("avg_tip")).alias("effect_size"))

# Examples
effect_size(df_tips1, "pickup_hour").show()
effect_size(df_tips1, "pickup_month").show()
effect_size(df_tips1, "is_weekend").show()
effect_size(df_tips1, "is_rush_hour").show()
effect_size(df_tips1, "is_night").show()

+------------------+
|       effect_size|
+------------------+
|5.7233674355406166|
+------------------+

+------------------+
|       effect_size|
+------------------+
|1.7177009408200554|
+------------------+

+------------------+
|       effect_size|
+------------------+
|0.9389339961051562|
+------------------+

+------------------+
|       effect_size|
+------------------+
|0.8663248268931589|
+------------------+

+-------------------+
|        effect_size|
+-------------------+
|0.19521329437030133|
+-------------------+



Turns out:

1. pickup_hour shows the largest spread → strong signal
2. pickup_month shows a small-to-moderate spread → useful but weaker
3. is_weekend / is_rush_hour / is_night → small effects, still useful as filters/controls but not primary drivers

In [29]:
from pyspark.sql.functions import avg, count, round

df_tips1.groupBy(
    "PULocationID", "DOLocationID", "pickup_hour", "pickup_month", "is_weekend", "is_rush_hour"
).agg(
    round(avg("tip_percent"), 2).alias("avg_tip_percent"),
    round(avg("tip_amount"), 2).alias("avg_tip_amount"),
    count("*").alias("rides")
).show()

+------------+------------+-----------+------------+----------+------------+---------------+--------------+-----+
|PULocationID|DOLocationID|pickup_hour|pickup_month|is_weekend|is_rush_hour|avg_tip_percent|avg_tip_amount|rides|
+------------+------------+-----------+------------+----------+------------+---------------+--------------+-----+
|          50|         186|          0|           1|         1|           0|          27.05|           2.7|    9|
|         236|         262|          0|           1|         1|           0|          44.68|          3.17|   45|
|          48|         260|          0|           1|         1|           0|          24.25|          7.15|    6|
|          90|         142|          0|           1|         1|           0|          21.21|          3.42|   45|
|           7|         116|          0|           1|         1|           0|          33.24|          9.14|    1|
|         229|         162|          1|           1|         1|           0|          32

Many of the rides here too less to give us a good average of tip percentage. Therefore we have created a function below that keeps dropping the lowest priority aggregations until the count of rides reaches at least 100.

In [40]:
from pyspark.sql.functions import col, avg, count, round

def get_avg_tip_percent_with_fallback(
    df,
    pu_location_id,
    do_location_id,
    pickup_hour,
    pickup_month,
    is_weekend,
    is_rush_hour,
    is_night,
    min_rides=100
):
    """
    Returns avg_tip_percent using fallback logic if rides are low.
    
    Priority of dropping filters (lowest priority dropped first):
    is_night -> is_rush_hour -> is_weekend -> pickup_month
    
    Output columns:
    avg_tip_percent, avg_tip_amount, rides_used
    """

    # Each step contains filters to apply
    # We start with the most strict filters
    fallback_steps = [
        {
            "fallback_level": "FULL (PU+DO+hour+month+weekend+rush+night)",
            "filters": [
                col("PULocationID") == pu_location_id,
                col("DOLocationID") == do_location_id,
                col("pickup_hour") == pickup_hour,
                col("pickup_month") == pickup_month,
                col("is_weekend") == is_weekend,
                col("is_rush_hour") == is_rush_hour,
                col("is_night") == is_night,
            ],
        },
        {
            "fallback_level": "DROP is_night",
            "filters": [
                col("PULocationID") == pu_location_id,
                col("DOLocationID") == do_location_id,
                col("pickup_hour") == pickup_hour,
                col("pickup_month") == pickup_month,
                col("is_weekend") == is_weekend,
                col("is_rush_hour") == is_rush_hour,
            ],
        },
        {
            "fallback_level": "DROP is_night + is_rush_hour",
            "filters": [
                col("PULocationID") == pu_location_id,
                col("DOLocationID") == do_location_id,
                col("pickup_hour") == pickup_hour,
                col("pickup_month") == pickup_month,
                col("is_weekend") == is_weekend,
            ],
        },
        {
            "fallback_level": "DROP is_night + is_rush_hour + is_weekend",
            "filters": [
                col("PULocationID") == pu_location_id,
                col("DOLocationID") == do_location_id,
                col("pickup_hour") == pickup_hour,
                col("pickup_month") == pickup_month,
            ],
        },
        {
            "fallback_level": "DROP is_night + is_rush_hour + is_weekend + pickup_month",
            "filters": [
                col("PULocationID") == pu_location_id,
                col("DOLocationID") == do_location_id,
                col("pickup_hour") == pickup_hour,
            ],
        },
    ]

    for step in fallback_steps:
        # Apply filters
        filtered_df = df
        for f in step["filters"]:
            filtered_df = filtered_df.filter(f)

        # Count rides
        rides = filtered_df.count()

        # If enough rides, calculate avg tip percent and return
        if rides >= min_rides:
            result = filtered_df.agg(
                round(avg("tip_percent"), 2).alias("avg_tip_percent"),
                round(avg("tip_amount"), 2).alias("avg_tip_amount"),
                count("*").alias("rides_used")
            )

            return result

    # If even the last fallback doesn't reach min rides, return whatever exists (even if low)
    from pyspark.sql.functions import lit
    last_df = df.filter(
        (col("PULocationID") == pu_location_id) &
        (col("DOLocationID") == do_location_id) &
        (col("pickup_hour") == pickup_hour)
    )

    return last_df.agg(
        round(avg("tip_percent"), 2).alias("avg_tip_percent"),
        round(avg("tip_amount"), 2).alias("avg_tip_amount"),
        count("*").alias("rides_used"),
    )

In [41]:
# Testing the function for a random set of values.

result_df = get_avg_tip_percent_with_fallback(
    df=df_tips1,
    pu_location_id=161,
    do_location_id=141,
    pickup_hour=20,
    pickup_month=1,
    is_weekend=1,
    is_rush_hour=0,
    is_night=1,
    min_rides=100
)

result_df.show(truncate=False)

+---------------+--------------+----------+
|avg_tip_percent|avg_tip_amount|rides_used|
+---------------+--------------+----------+
|27.23          |2.91          |974       |
+---------------+--------------+----------+



### Nearby hotspots

In [16]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg
from pyspark.sql.window import Window

# Calculate total pickups for each hour (denominator for probability)
total_pickups_by_hour = df_tips1.groupBy("pickup_hour") \
    .agg(count("*").alias("total_pickups_in_hour"))

# Calculate pickups by location and hour
pickups_by_location_hour = df_tips1.groupBy("PULocationID", "pickup_hour") \
    .agg(count("*").alias("location_pickups"))

# Join to calculate probability
location_probabilities = pickups_by_location_hour \
    .join(total_pickups_by_hour, "pickup_hour") \
    .withColumn(
        "pickup_probability_pct",
        (col("location_pickups") / col("total_pickups_in_hour")) * 100
    ) \
    .select(
        "PULocationID",
        "pickup_hour",
        "location_pickups",
        "total_pickups_in_hour",
        "pickup_probability_pct"
    )

# This is conditional probability - Given it is hour = h, what % of rides happen in the pickup zone = z?

print("=== Location Pickup Probabilities ===")
location_probabilities.orderBy(col("pickup_probability_pct").desc()).show(20, truncate=False)

=== Location Pickup Probabilities ===
+------------+-----------+----------------+---------------------+----------------------+
|PULocationID|pickup_hour|location_pickups|total_pickups_in_hour|pickup_probability_pct|
+------------+-----------+----------------+---------------------+----------------------+
|79          |2          |110090          |943524               |11.667959691539378    |
|79          |3          |69117           |593998               |11.635897763965536    |
|132         |5          |42157           |407860               |10.336144755553377    |
|79          |1          |149881          |1455480              |10.297702476159069    |
|148         |3          |57584           |593998               |9.694308735046246     |
|132         |6          |95555           |1009596              |9.464676959892868     |
|148         |2          |86542           |943524               |9.172209715916075     |
|249         |2          |81176           |943524               |8.60349

In [29]:
# Method 1: Based on actual trips (where passengers go FROM the current location)
# This shows connected locations based on real trip data

def get_connected_locations(df):
    """
    Find locations that are connected based on trip patterns.
    If many passengers go from A to B, then B is 'nearby' A
    """
    connections = df.groupBy("PULocationID", "DOLocationID") \
        .agg(count("*").alias("connection_count")) \
        .filter(col("connection_count") >= 5)  # Minimum threshold
    
    # Calculate connection strength (what % of trips from location A go to location B)
    window_spec = Window.partitionBy("PULocationID")
    
    connections_with_pct = connections.withColumn(
        "total_from_location",
        spark_sum("connection_count").over(window_spec)
    ).withColumn(
        "connection_strength_pct",
        (col("connection_count") / col("total_from_location")) * 100
    )
    
    return connections_with_pct

connected_locations = get_connected_locations(df_tips1)

print("=== Location Connections ===")
connected_locations.orderBy(col("connection_strength_pct").desc()).show(20, truncate=False)

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: functions does not exist in the JVM

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [19]:
from pyspark.sql.functions import rank, dense_rank

def create_nearby_hotspot_probabilities(df):
    """
    For each location and hour, find nearby locations with their pickup probabilities
    """
    # Get connections (nearby locations)
    connections = get_connected_locations(df)
    
    # Get pickup probabilities
    pickups_by_location_hour = df.groupBy("PULocationID", "pickup_hour") \
        .agg(count("*").alias("location_pickups"))
    
    total_pickups_by_hour = df.groupBy("pickup_hour") \
        .agg(count("*").alias("total_pickups_in_hour"))
    
    location_probs = pickups_by_location_hour \
        .join(total_pickups_by_hour, "pickup_hour") \
        .withColumn(
            "pickup_probability_pct",
            (col("location_pickups") / col("total_pickups_in_hour")) * 100
        )
    
    # Join connections with probabilities
    # Current location -> Connected location -> Probability at that location
    nearby_hotspot_probs = connections.alias("conn") \
        .join(
            location_probs.alias("probs"),
            col("conn.DOLocationID") == col("probs.PULocationID"),
            "inner"
        ) \
        .select(
            col("conn.PULocationID").alias("current_location"),
            col("conn.DOLocationID").alias("nearby_location"),
            col("conn.connection_count").alias("trips_between"),
            col("conn.connection_strength_pct").alias("connection_strength_pct"),
            col("probs.pickup_hour"),
            col("probs.location_pickups").alias("nearby_pickups"),
            col("probs.pickup_probability_pct").alias("nearby_pickup_probability_pct")
        )
    
    # Calculate combined probability score
    # This considers both: how connected the locations are AND pickup probability
    nearby_hotspot_probs = nearby_hotspot_probs.withColumn(
        "combined_probability_score",
        col("connection_strength_pct") * col("nearby_pickup_probability_pct") / 100
    )
    
    return nearby_hotspot_probs

# Create the probabilities
nearby_probs = create_nearby_hotspot_probabilities(df_tips1)

print("=== Nearby Hotspot Probabilities ===")
nearby_probs.orderBy(col("combined_probability_score").desc()).show(20, truncate=False)

=== Nearby Hotspot Probabilities ===
+----------------+---------------+-------------+-----------------------+-----------+--------------+-----------------------------+--------------------------+
|current_location|nearby_location|trips_between|connection_strength_pct|pickup_hour|nearby_pickups|nearby_pickup_probability_pct|combined_probability_score|
+----------------+---------------+-------------+-----------------------+-----------+--------------+-----------------------------+--------------------------+
|216             |132            |7553         |35.468419816858415     |5          |42157         |10.336144755553377           |3.6660672147778657        |
|134             |132            |2883         |34.86094316807739      |5          |42157         |10.336144755553377           |3.6032775490036744        |
|216             |132            |7553         |35.468419816858415     |6          |95555         |9.464676959892868            |3.356971358444275         |
|134             |132

In [21]:
# Save
nearby_probs.write.mode("overwrite") \
    .parquet("nearby_hotspot_probabilities")

In [23]:
def get_driver_recommendations(current_location, current_hour, top_n=5):
    """
    Get top N nearby locations with normalized probabilities that sum to 100%
    """
    # Load pre-computed data
    nearby_probs = spark.read.parquet("nearby_hotspot_probabilities")
    
    # Filter for current location and hour
    filtered = nearby_probs.filter(
        (col("current_location") == current_location) &
        (col("pickup_hour") == current_hour)
    ).orderBy(col("combined_probability_score").desc()).limit(top_n)
    
    # Calculate the sum of scores for normalization
    filtered_pd = filtered.toPandas()
    
    if len(filtered_pd) > 0:
        total_score = filtered_pd['combined_probability_score'].sum()
        filtered_pd['probability_pct'] = (filtered_pd['combined_probability_score'] / total_score) * 100
        
        # Convert back to Spark if needed, or just use pandas for display
        result = filtered_pd[[
            'nearby_location',
            'nearby_pickups',
            'nearby_pickup_probability_pct',
            'connection_strength_pct',
            'probability_pct'
        ]].round(2)
        
        return result
    else:
        return None

# Test the function
print("\n=== Driver Recommendations ===")
print("Driver at Location 161, Hour 18 (6 PM)")
print("\nTop 5 Nearby Hotspots (by probability):\n")

recommendations = get_driver_recommendations(current_location=161, current_hour=18, top_n=5)
if recommendations is not None:
    print(recommendations.to_string(index=False))
else:
    print("No recommendations available for this location/time")


=== Driver Recommendations ===
Driver at Location 161, Hour 18 (6 PM)

Top 5 Nearby Hotspots (by probability):

 nearby_location  nearby_pickups  nearby_pickup_probability_pct  connection_strength_pct  probability_pct
             237          307716                           5.29                     7.20            33.30
             236          266523                           4.58                     6.30            25.22
             161          359923                           6.18                     3.54            19.13
             234          188752                           3.24                     4.29            12.17
             163          211559                           3.63                     3.20            10.17


In [25]:
def get_enhanced_recommendations(current_location, current_hour, top_n=5):
    """
    Enhanced recommendations with fare and distance info
    """
    nearby_probs = spark.read.parquet("nearby_hotspot_probabilities")
    
    # Get average fare and distance for trips between locations
    trip_details = df.groupBy("PULocationID", "DOLocationID") \
        .agg(
            avg("trip_distance").alias("avg_distance_to_location"),
            avg("fare_amount").alias("avg_fare_to_location"),
            avg("tip_amount").alias("avg_tip_to_location")
        )
    
    # Join with probabilities
    enhanced = nearby_probs.filter(
        (col("current_location") == current_location) &
        (col("pickup_hour") == current_hour)
    ).join(
        trip_details,
        (nearby_probs["current_location"] == trip_details["PULocationID"]) &
        (nearby_probs["nearby_location"] == trip_details["DOLocationID"]),
        "left"
    ).select(
        col("nearby_location"),
        col("nearby_pickups"),
        col("nearby_pickup_probability_pct"),
        col("connection_strength_pct"),
        col("combined_probability_score"),
        col("avg_distance_to_location"),
        col("avg_fare_to_location")
    ).orderBy(col("combined_probability_score").desc()).limit(top_n)
    
    # Convert to pandas for final processing
    enhanced_pd = enhanced.toPandas()
    
    if len(enhanced_pd) > 0:
        # Normalize probabilities to sum to 100%
        total_score = enhanced_pd['combined_probability_score'].sum()
        enhanced_pd['probability_pct'] = (enhanced_pd['combined_probability_score'] / total_score) * 100
        
        # Clean up and format
        result = enhanced_pd[[
            'nearby_location',
            'probability_pct',
            'nearby_pickups',
            'avg_distance_to_location',
            'avg_fare_to_location'
        ]].round(2)
        
        result.columns = [
            'Location ID',
            'Probability (%)',
            'Pickups/Hour',
            'Distance (mi)',
            'Avg Fare ($)'
        ]
        
        return result
    else:
        return None

# Test enhanced recommendations
print("\n=== Enhanced Driver Recommendations ===")
print("Driver at Location 161, Hour 18 (6 PM)\n")

recommendations = get_enhanced_recommendations(current_location=161, current_hour=18, top_n=5)
if recommendations is not None:
    print(recommendations.to_string(index=False))
    print(f"\nTotal Probability: {recommendations['Probability (%)'].sum():.1f}%")


=== Enhanced Driver Recommendations ===
Driver at Location 161, Hour 18 (6 PM)

 Location ID  Probability (%)  Pickups/Hour  Distance (mi)  Avg Fare ($)
         237            33.30        307716           1.07         10.24
         236            25.22        266523           1.94         14.22
         161            19.13        359923           0.60          9.07
         234            12.17        188752           1.44         12.05
         163            10.17        211559           0.79          9.42

Total Probability: 100.0%


In [26]:
def get_recommendations_with_time_window(current_location, current_hour, window_hours=2, top_n=5):
    """
    Look at next N hours to give better recommendations
    """
    nearby_probs = spark.read.parquet("nearby_hotspot_probabilities")
    
    # Get hours in window (handle wraparound)
    hours_to_check = [(current_hour + i) % 24 for i in range(window_hours)]
    
    # Filter for location and time window
    filtered = nearby_probs.filter(
        (col("current_location") == current_location) &
        (col("pickup_hour").isin(hours_to_check))
    )
    
    # Aggregate by nearby location (sum across hours)
    aggregated = filtered.groupBy("nearby_location") \
        .agg(
            spark_sum("nearby_pickups").alias("total_pickups_in_window"),
            avg("nearby_pickup_probability_pct").alias("avg_probability_pct"),
            avg("combined_probability_score").alias("avg_combined_score")
        ) \
        .orderBy(col("avg_combined_score").desc()) \
        .limit(top_n)
    
    # Convert and normalize
    result_pd = aggregated.toPandas()
    
    if len(result_pd) > 0:
        total_score = result_pd['avg_combined_score'].sum()
        result_pd['probability_pct'] = (result_pd['avg_combined_score'] / total_score) * 100
        
        result = result_pd[[
            'nearby_location',
            'probability_pct',
            'total_pickups_in_window',
            'avg_probability_pct'
        ]].round(2)
        
        result.columns = [
            'Location ID',
            'Probability (%)',
            f'Pickups (next {window_hours}hrs)',
            'Avg Hourly Prob (%)'
        ]
        
        return result
    else:
        return None

# Test with time window
print("\n=== Recommendations for Next 2 Hours ===")
print("Driver at Location 161, starting at 6 PM\n")

recommendations = get_recommendations_with_time_window(
    current_location=161, 
    current_hour=18, 
    window_hours=2, 
    top_n=5
)
if recommendations is not None:
    print(recommendations.to_string(index=False))
    print(f"\nTotal Probability: {recommendations['Probability (%)'].sum():.1f}%")


=== Recommendations for Next 2 Hours ===
Driver at Location 161, starting at 6 PM

 Location ID  Probability (%)  Pickups (next 2hrs)  Avg Hourly Prob (%)
         237            33.20               556757                 5.05
         236            23.85               459182                 4.15
         161            19.65               669698                 6.09
         234            13.05               365711                 3.33
         163            10.25               386516                 3.51

Total Probability: 100.0%


In [28]:
def driver_hotspot_dashboard(current_location, current_hour, top_n=5):
    """
    Complete dashboard function with all metrics
    """
    print(f"\n{'='*70}")
    print(f"DRIVER HOTSPOT RECOMMENDATIONS")
    print(f"{'='*70}")
    print(f"Current Location: {current_location}")
    print(f"Current Time: {current_hour}:00 ({current_hour % 12 or 12} {'PM' if current_hour >= 12 else 'AM'})")
    print(f"{'='*70}\n")
    
    # Get recommendations
    recommendations = get_enhanced_recommendations(current_location, current_hour, top_n)
    
    if recommendations is not None:
        print("TOP NEARBY HOTSPOTS (by probability of finding next ride):\n")
        
        for idx, row in recommendations.iterrows():
            print(f"{idx + 1}. Location {int(row['Location ID'])}")
            print(f"   Probability: {row['Probability (%)']:.1f}%")
            print(f"   Expected Pickups: ~{int(row['Pickups/Hour'])} rides/hour")
            print(f"   Distance: {row['Distance (mi)']:.1f} miles away")
            print(f"   Avg Fare: ${row['Avg Fare ($)']:.2f}")
            print()
        
        print(f"Total Coverage: {recommendations['Probability (%)'].sum():.1f}% of nearby opportunities")
    else:
        print("⚠ No recommendations available for this location/time combination")
        print("Consider trying a nearby location or different time.")
    
    print(f"\n{'='*70}")

# Test the dashboard
driver_hotspot_dashboard(current_location=161, current_hour=18, top_n=5)
# ```

# ## **Output Example:**
# ```
# ======================================================================
# DRIVER HOTSPOT RECOMMENDATIONS
# ======================================================================
# Current Location: 161
# Current Time: 18:00 (6 PM)
# ======================================================================

# TOP NEARBY HOTSPOTS (by probability of finding next ride):

# 1. Location 237
#    Probability: 32.5%
#    Expected Pickups: ~67 rides/hour
#    Distance: 1.2 miles away
#    Avg Fare: $18.50

# 2. Location 142
#    Probability: 24.8%
#    Expected Pickups: ~54 rides/hour
#    Distance: 0.8 miles away
#    Avg Fare: $22.30

# 3. Location 236
#    Probability: 18.3%
#    Expected Pickups: ~45 rides/hour
#    Distance: 1.5 miles away
#    Avg Fare: $15.75

# 4. Location 141
#    Probability: 15.2%
#    Expected Pickups: ~38 rides/hour
#    Distance: 0.5 miles away
#    Avg Fare: $12.40

# 5. Location 138
#    Probability: 9.2%
#    Expected Pickups: ~28 rides/hour
#    Distance: 2.1 miles away
#    Avg Fare: $25.60

# Total Coverage: 100.0% of nearby opportunities

# ======================================================================


DRIVER HOTSPOT RECOMMENDATIONS
Current Location: 161
Current Time: 18:00 (6 PM)

TOP NEARBY HOTSPOTS (by probability of finding next ride):

1. Location 237
   Probability: 33.3%
   Expected Pickups: ~307716 rides/hour
   Distance: 1.1 miles away
   Avg Fare: $10.24

2. Location 236
   Probability: 25.2%
   Expected Pickups: ~266523 rides/hour
   Distance: 1.9 miles away
   Avg Fare: $14.22

3. Location 161
   Probability: 19.1%
   Expected Pickups: ~359923 rides/hour
   Distance: 0.6 miles away
   Avg Fare: $9.07

4. Location 234
   Probability: 12.2%
   Expected Pickups: ~188752 rides/hour
   Distance: 1.4 miles away
   Avg Fare: $12.05

5. Location 163
   Probability: 10.2%
   Expected Pickups: ~211559 rides/hour
   Distance: 0.8 miles away
   Avg Fare: $9.42

Total Coverage: 100.0% of nearby opportunities



### Nearby Hotspots 2

In [13]:
# Load the zone lookup table
zone_lookup = spark.read.csv("s3a://nyc-taxi/taxi_zone_lookup.csv", header=True, inferSchema=True)

print("=== Zone Lookup Table Schema ===")
zone_lookup.printSchema()

print("\n=== Sample Zone Data ===")
zone_lookup.show(20, truncate=False)

print("\n=== Boroughs Available ===")
zone_lookup.select("Borough").distinct().show(truncate=False)

from pyspark.sql.functions import desc

# Now you can use desc() directly
zone_lookup.groupBy("Borough").count().orderBy(desc("count")).show()

=== Zone Lookup Table Schema ===
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)


=== Sample Zone Data ===
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10  

#### 2 Borough entries include Unknown and N/A, so we will filter them out.

In [14]:
# Define the boroughs to exclude
exclude_boroughs = ["Unknown", "N/A"]

# Filter the zone_lookup table to get the clean IDs
df_clean_zones = zone_lookup.filter(~col("Borough").isin(exclude_boroughs))

# If you want to filter your main trip data (df_tips1) 
# using the clean zone list:
valid_ids = [row['LocationID'] for row in df_clean_zones.select("LocationID").collect()]

df_tips_filtered = df_tips1.filter(col("PULocationID").isin(valid_ids) & 
                                  col("DOLocationID").isin(valid_ids))
df_tips1.count()

79757391

In [23]:
df_tips_filtered.count()

78879710

In [24]:
df_clean_zones.groupBy("Borough").count().orderBy(desc("count")).show()

+-------------+-----+
|      Borough|count|
+-------------+-----+
|       Queens|   69|
|    Manhattan|   69|
|     Brooklyn|   61|
|        Bronx|   43|
|Staten Island|   20|
|          EWR|    1|
+-------------+-----+



EWR (Newark Airport) is physically separated from the other boroughs by a significant distance and a state line. Combining it with a nearby borough like Manhattan or Queens would create a "ghost hotspot" that a driver cannot realistically reach without a long, expensive trip.

So, removing EWR is the much better choice for your taxi driver dashboard rather than combining it.

In [15]:
df_clean_zones = df_clean_zones.filter(~col("Borough").isin(["EWR"]))

# If you want to filter your main trip data (df_tips1) 
# using the clean zone list:
valid_ids = [row['LocationID'] for row in df_clean_zones.select("LocationID").collect()]

df_tips_filtered = df_tips_filtered.filter(col("PULocationID").isin(valid_ids) & 
                                  col("DOLocationID").isin(valid_ids))

In [26]:
df_tips1.count()

79757391

In [16]:
df_tips_filtered.count()

78664614

#### Create Borough-Based Nearby Locations

In [17]:
from pyspark.sql.functions import col, count, avg, sum as spark_sum
from pyspark.sql.window import Window

# Join your trip data with zone lookup to get borough information
df_with_borough = df_tips_filtered.join(
    zone_lookup.select(
        col("LocationID").alias("PULocationID"),
        col("Borough").alias("PU_Borough"),
        col("Zone").alias("PU_Zone")
    ),
    "PULocationID",
    "left"
)

print("=== Trip Data with Borough Info ===")
df_with_borough.select("PULocationID", "PU_Borough", "PU_Zone", "pickup_hour").show(20, truncate=False)

# Check for any missing borough mappings
print("\n=== Checking for Missing Borough Mappings ===")
missing_mappings = df_with_borough.filter(col("PU_Borough").isNull())
print(f"Trips with missing borough: {missing_mappings.count()}")

=== Trip Data with Borough Info ===
+------------+----------+-----------------------+-----------+
|PULocationID|PU_Borough|PU_Zone                |pickup_hour|
+------------+----------+-----------------------+-----------+
|43          |Manhattan |Central Park           |0          |
|48          |Manhattan |Clinton East           |0          |
|138         |Queens    |LaGuardia Airport      |0          |
|107         |Manhattan |Gramercy               |0          |
|161         |Manhattan |Midtown Center         |0          |
|239         |Manhattan |Upper West Side South  |0          |
|142         |Manhattan |Lincoln Square East    |0          |
|164         |Manhattan |Midtown South          |0          |
|234         |Manhattan |Union Sq               |0          |
|164         |Manhattan |Midtown South          |0          |
|138         |Queens    |LaGuardia Airport      |0          |
|33          |Brooklyn  |Brooklyn Heights       |0          |
|79          |Manhattan |East Vill

#### Calculate Pickup Probabilities by Borough

In [18]:
# Calculate pickups by location, borough, and hour
pickups_by_location_hour = df_with_borough.groupBy("PULocationID", "PU_Borough", "pickup_hour") \
    .agg(
        count("*").alias("location_pickups"),
        avg("fare_amount").alias("avg_fare"),
        avg("trip_distance").alias("avg_distance"),
        avg("tip_percent").alias("avg_tip_percent")
    )

# Calculate total pickups in the borough for each hour
total_pickups_by_borough_hour = df_with_borough.groupBy("PU_Borough", "pickup_hour") \
    .agg(count("*").alias("total_borough_pickups"))

# Join to calculate probability within borough
location_probabilities_in_borough = pickups_by_location_hour \
    .join(
        total_pickups_by_borough_hour,
        ["PU_Borough", "pickup_hour"],
        "inner"
    ) \
    .withColumn(
        "pickup_probability_in_borough_pct",
        (col("location_pickups") / col("total_borough_pickups")) * 100
    ) \
    .select(
        "PULocationID",
        "PU_Borough",
        "pickup_hour",
        "location_pickups",
        "total_borough_pickups",
        "pickup_probability_in_borough_pct",
        "avg_fare",
        "avg_distance",
        "avg_tip_percent"
    )

# pickup_probability_in_borough_pct calculates: "Given that a pickup happens in this borough at this hour, what's the probability it happens at this specific location?"

print("=== Location Probabilities within Borough ===")
location_probabilities_in_borough.orderBy(col("pickup_probability_in_borough_pct").desc()).show(20, truncate=False)

=== Location Probabilities within Borough ===
+------------+-------------+-----------+----------------+---------------------+---------------------------------+------------------+------------------+-------------------+
|PULocationID|PU_Borough   |pickup_hour|location_pickups|total_borough_pickups|pickup_probability_in_borough_pct|avg_fare          |avg_distance      |avg_tip_percent    |
+------------+-------------+-----------+----------------+---------------------+---------------------------------+------------------+------------------+-------------------+
|23          |Staten Island|6          |100             |123                  |81.30081300813008                |63.46200000000001 |24.068100000000005|0.22987390882638217|
|214         |Staten Island|5          |114             |150                  |76.0                             |60.78333333333333 |19.52026315789474 |1.609034088288205  |
|132         |Queens       |6          |90804           |119999               |75.670630588588

In [20]:
# Save this
location_probabilities_in_borough.write.mode("overwrite") \
    .parquet("location_probabilities_in_borough")

#### Create Nearby Hotspots Based on Same Borough

In [ ]:
def get_nearby_hotspots_by_borough(current_location, current_hour, top_n=5):
    """
    Find top hotspots in the SAME BOROUGH as the current location
    """
    # Load zone lookup
    zone_lookup = spark.read.csv("s3a://nyc-taxi/taxi_zone_lookup.csv", header=True, inferSchema=True)
    
    # Load location probabilities
    location_probs = spark.read.parquet("location_probabilities_in_borough")
    
    # Get the borough of the current location
    current_borough = zone_lookup.filter(col("LocationID") == current_location) \
        .select("Borough").first()
    
    if current_borough is None:
        print(f"Location {current_location} not found in zone lookup")
        return None
    
    current_borough_name = current_borough["Borough"]
    print(f"Current Borough: {current_borough_name}")
    
    # Get all locations in the same borough at the current hour
    same_borough_hotspots = location_probs.filter(
        (col("PU_Borough") == current_borough_name) &
        (col("pickup_hour") == current_hour)
    ).orderBy(col("pickup_probability_in_borough_pct").desc())
    
    # Get top N locations (excluding current location)
    top_hotspots = same_borough_hotspots.filter(col("PULocationID") != current_location).limit(top_n)
    
    return top_hotspots

# Test the function
print("\n=== Nearby Hotspots (Same Borough) ===")
print("Driver at Location 161, Hour 18 (6 PM)\n")

hotspots = get_nearby_hotspots_by_borough(current_location=161, current_hour=18, top_n=5)
if hotspots:
    hotspots.show(truncate=False)

In [21]:
def get_enhanced_nearby_hotspots(current_location, current_hour, top_n=5):
    """
    Enhanced version with zone names and better formatting
    """
    # Load data
    zone_lookup = spark.read.csv("s3a://nyc-taxi/taxi_zone_lookup.csv", header=True, inferSchema=True)
    location_probs = spark.read.parquet("location_probabilities_in_borough")
    
    # Get current location info
    current_zone_info = zone_lookup.filter(col("LocationID") == current_location).first()
    
    if current_zone_info is None:
        print(f"Location {current_location} not found")
        return None
    
    current_borough = current_zone_info["Borough"]
    current_zone = current_zone_info["Zone"]
    
    print(f"Current Location: {current_zone} ({current_borough})")
    print(f"Current Time: {current_hour}:00\n")
    
    # Get hotspots in same borough
    same_borough_hotspots = location_probs.filter(
        (col("PU_Borough") == current_borough) &
        (col("pickup_hour") == current_hour) &
        (col("PULocationID") != current_location)
    )
    
    # Join with zone lookup to get zone names
    hotspots_with_names = same_borough_hotspots.join(
        zone_lookup.select(
            col("LocationID").alias("PULocationID"),
            col("Zone").alias("zone_name")
        ),
        "PULocationID",
        "left"
    ).select(
        "PULocationID",
        "zone_name",
        "location_pickups",
        "pickup_probability_in_borough_pct",
        "avg_fare",
        "avg_distance",
        "avg_tip_percent"
    ).orderBy(col("pickup_probability_in_borough_pct").desc()).limit(top_n)
    
    return hotspots_with_names

# Test enhanced version
print("=== Enhanced Nearby Hotspots ===\n")
hotspots = get_enhanced_nearby_hotspots(current_location=161, current_hour=18, top_n=5)
if hotspots:
    hotspots.show(truncate=False)

=== Enhanced Nearby Hotspots ===

Current Location: Midtown Center (Manhattan)
Current Time: 18:00

+------------+---------------------+----------------+---------------------------------+------------------+------------------+------------------+
|PULocationID|zone_name            |location_pickups|pickup_probability_in_borough_pct|avg_fare          |avg_distance      |avg_tip_percent   |
+------------+---------------------+----------------+---------------------------------+------------------+------------------+------------------+
|237         |Upper East Side South|306718          |5.792690872788491                |12.772591631400843|1.713306294381158 |28.725783173835083|
|162         |Midtown East         |269887          |5.097098838621364                |15.01047727382199 |2.0333778951931736|26.75317617408619 |
|236         |Upper East Side North|265696          |5.017947411421602                |12.928201628929326|1.8291468068770318|28.730728518766   |
|163         |Midtown North   

In [25]:
location_probabilities_in_borough.show()

+------------+----------+-----------+----------------+---------------------+---------------------------------+------------------+------------------+-------------------+
|PULocationID|PU_Borough|pickup_hour|location_pickups|total_borough_pickups|pickup_probability_in_borough_pct|          avg_fare|      avg_distance|    avg_tip_percent|
+------------+----------+-----------+----------------+---------------------+---------------------------------+------------------+------------------+-------------------+
|          67|  Brooklyn|         13|             101|                38478|               0.2624876552835386| 31.96237623762376| 7.106435643564355|                0.0|
|          77|  Brooklyn|         13|             513|                38478|               1.3332293778262905|31.418771929824562|7.1061793372319695| 0.2094647179090035|
|         225|  Brooklyn|         13|            1066|                38478|               2.7704142626955663|28.867270168855534| 6.195712945590995| 0.4878

In [26]:
def get_normalized_probabilities(current_location, current_hour, top_n=5):
    """
    Get top N locations with probabilities normalized to sum to 100%
    """
    # Load data
    zone_lookup = spark.read.csv("s3a://nyc-taxi/taxi_zone_lookup.csv", header=True, inferSchema=True)
    location_probs = spark.read.parquet("location_probabilities_in_borough")
    
    # Get current borough
    current_zone_info = zone_lookup.filter(col("LocationID") == current_location).first()
    
    if current_zone_info is None:
        return None
    
    current_borough = current_zone_info["Borough"]
    
    # Get hotspots
    hotspots = location_probs.filter(
        (col("PU_Borough") == current_borough) &
        (col("pickup_hour") == current_hour) &
        (col("PULocationID") != current_location)
    ).orderBy(col("pickup_probability_in_borough_pct").desc()).limit(top_n)
    
    # Join with zone names
    hotspots_with_names = hotspots.join(
        zone_lookup.select(
            col("LocationID").alias("PULocationID"),
            col("Zone").alias("zone_name")
        ),
        "PULocationID",
        "left"
    )
    
    # Convert to pandas for normalization
    hotspots_pd = hotspots_with_names.toPandas()
    
    if len(hotspots_pd) > 0:
        # Normalize probabilities to sum to 100%
        total_prob = hotspots_pd['pickup_probability_in_borough_pct'].sum()
        hotspots_pd['normalized_probability_pct'] = (
            hotspots_pd['pickup_probability_in_borough_pct'] / total_prob * 100
        )
        
        # Format for display
        result = hotspots_pd[[
            'PULocationID',
            'zone_name',
            'normalized_probability_pct',
            'location_pickups',
            'avg_fare',
            'avg_tip_percent'
        ]].round(2)
        
        result.columns = [
            'Location ID',
            'Zone Name',
            'Probability (%)',
            'Pickups/Hour',
            'Avg Fare ($)',
            'Avg Tip (%)'
        ]
        
        return result
    else:
        return None

# Test normalized probabilities
print("\n=== Normalized Probability Recommendations ===")
print("Driver at Location 161, Hour 18 (6 PM)\n")

recommendations = get_normalized_probabilities(current_location=161, current_hour=18, top_n=5)
if recommendations is not None:
    print(recommendations.to_string(index=False))
    print(f"\nTotal Probability: {recommendations['Probability (%)'].sum():.1f}%")


=== Normalized Probability Recommendations ===
Driver at Location 161, Hour 18 (6 PM)

 Location ID             Zone Name  Probability (%)  Pickups/Hour  Avg Fare ($)  Avg Tip (%)
         237 Upper East Side South            24.47        306718         12.77        28.73
         162          Midtown East            21.53        269887         15.01        26.75
         236 Upper East Side North            21.20        265696         12.93        28.73
         163         Midtown North            16.80        210563         14.72        27.27
         142   Lincoln Square East            16.01        200691         13.21        28.45

Total Probability: 100.0%


In [27]:
def driver_dashboard_by_borough(current_location, current_hour, top_n=5):
    """
    Complete dashboard showing nearby hotspots within the same borough
    """
    # Load data
    zone_lookup = spark.read.csv("s3a://nyc-taxi/taxi_zone_lookup.csv", header=True, inferSchema=True)
    
    # Get current location details
    current_info = zone_lookup.filter(col("LocationID") == current_location).first()
    
    if current_info is None:
        print(f"⚠ Location {current_location} not found in zone lookup table")
        return
    
    current_borough = current_info["Borough"]
    current_zone = current_info["Zone"]
    
    print(f"\n{'='*80}")
    print(f"DRIVER HOTSPOT RECOMMENDATIONS (Borough-Based)")
    print(f"{'='*80}")
    print(f"Your Location: {current_zone} (Zone ID: {current_location})")
    print(f"Borough: {current_borough}")
    print(f"Time: {current_hour}:00 ({current_hour % 12 or 12} {'PM' if current_hour >= 12 else 'AM'})")
    print(f"{'='*80}\n")
    
    # Get recommendations
    recommendations = get_normalized_probabilities(current_location, current_hour, top_n)
    
    if recommendations is not None and len(recommendations) > 0:
        print(f"TOP {len(recommendations)} HOTSPOTS IN {current_borough.upper()}:\n")
        
        for idx, row in recommendations.iterrows():
            print(f"{idx + 1}. {row['Zone Name']} (ID: {int(row['Location ID'])})")
            print(f"   📊 Probability: {row['Probability (%)']:.1f}% of finding next ride here")
            print(f"   🚕 Expected Activity: ~{int(row['Pickups/Hour'])} pickups/hour")
            print(f"   💰 Average Fare: ${row['Avg Fare ($)']:.2f}")
            print(f"   💵 Average Tip: {row['Avg Tip (%)']:.1f}%")
            print()
        
        print(f"✅ These {len(recommendations)} locations represent {recommendations['Probability (%)'].sum():.1f}% ")
        print(f"   of pickup opportunities in {current_borough}")
    else:
        print(f"⚠ No recommendations available for {current_borough} at this time")
        print("   Consider checking a different hour or nearby borough")
    
    print(f"\n{'='*80}")

# Test the complete dashboard
driver_dashboard_by_borough(current_location=161, current_hour=18, top_n=5)
driver_dashboard_by_borough(current_location=161, current_hour=8, top_n=5)


DRIVER HOTSPOT RECOMMENDATIONS (Borough-Based)
Your Location: Midtown Center (Zone ID: 161)
Borough: Manhattan
Time: 18:00 (6 PM)

TOP 5 HOTSPOTS IN MANHATTAN:

1. Upper East Side South (ID: 237)
   📊 Probability: 24.5% of finding next ride here
   🚕 Expected Activity: ~306718 pickups/hour
   💰 Average Fare: $12.77
   💵 Average Tip: 28.7%

2. Midtown East (ID: 162)
   📊 Probability: 21.5% of finding next ride here
   🚕 Expected Activity: ~269887 pickups/hour
   💰 Average Fare: $15.01
   💵 Average Tip: 26.8%

3. Upper East Side North (ID: 236)
   📊 Probability: 21.2% of finding next ride here
   🚕 Expected Activity: ~265696 pickups/hour
   💰 Average Fare: $12.93
   💵 Average Tip: 28.7%

4. Midtown North (ID: 163)
   📊 Probability: 16.8% of finding next ride here
   🚕 Expected Activity: ~210563 pickups/hour
   💰 Average Fare: $14.72
   💵 Average Tip: 27.3%

5. Lincoln Square East (ID: 142)
   📊 Probability: 16.0% of finding next ride here
   🚕 Expected Activity: ~200691 pickups/hour
   